[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/34.%20The%20Hazard%20%26%20IMDG%20Segregation%20Problem/P34-Tier-5_executed.ipynb)

---


# 34. The Hazard and IMDG Segregation Problem

## Tier 5: Integrated Digital-Twin Simulation

### Goal

This notebook builds a small, illustrative digital-twin simulation for dangerous-goods operations. It connects a simplified vessel-motion model, environmental conditions, container sensor readings, a heuristic risk assessment, and what-if decision support. The purpose is to show how these components can be connected in a system-of-systems workflow; the simulation is not a validated maritime safety system.

### Inputs and outputs

The inputs are a synthetic set of containers, vessel positions, segregation requirements, and stochastic environmental and motion updates. The simulation produces a time series of risk assessments, summary statistics, visualizations, and—when the assessed risk is High or Critical—cost comparisons for four response scenarios.

### Model assumptions and scope

- Container classes, positions, sensor readings, and segregation requirements form a simplified teaching scenario. They are not a complete or authoritative IMDG Code implementation.
- Coordinates are measured in metres. Distance-based requirements use three-dimensional Euclidean distance between position coordinates. Requirements marked `different_hold` instead use the hold identifiers.
- Initial placement is a simple randomized demonstration heuristic, not an optimization procedure. It does not guarantee that every segregation requirement is satisfied.
- Vessel motion, weather, sensor behaviour, and risk scores are stylized simulation rules. The risk score is a weighted heuristic, not a statistically calibrated probability of an actual incident. The time-to-failure value is also an illustrative random value for risk bands, not a forecast supported by survival data.
- The simulator includes a cargo-force calculation and a weather-forecast generator as component examples. The current synchronization cycle does not use either method to update the risk assessment.
- Scenario costs and risk reductions are illustrative decision-support inputs. A risk reduction is an absolute decrease in failure-probability points, not a relative percentage reduction. The incident-cost assumption is used to compare expected costs; it is not an empirical estimate.
- Stochastic draws are not seeded, so repeated executions can produce different trajectories. The simulation retains at most the most recent 1,000 synchronization records; whole-run cycle counts and maximum risk are tracked separately.

### Workflow

1. Define the data structures for containers, positions, requirements, vessel state, and risk assessments.
2. Create the synthetic scenario.
3. Update vessel and environmental state, then update container sensor readings.
4. Combine four heuristic risk components into a weighted overall score and risk category.
5. Generate response scenarios for High or Critical assessments and compare their expected costs.
6. Inspect the retained simulation history and summarize the system's behaviour.

### Interpreting the results

The outputs describe the behaviour of the stated simulation rules under one stochastic run. They should not be interpreted as operational forecasts, proof of regulatory compliance, or evidence that a particular response is safe. In particular, the reported risk score and confidence value are model indicators, not calibrated probabilities or statistical confidence intervals.

In [ ]:
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from itertools import combinations
from typing import Any, Dict, List, Optional
import random

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.patches import Patch

plt.style.use("default")
sns.set_palette("husl")

# This is a scenario assumption used to compare response options, not an observed loss.
INCIDENT_COST = 2_300_000


In [ ]:
@dataclass
class Container:
    """A scenario container with illustrative sensor readings."""
    id: str
    imdg_class: str
    weight: float = 20.0
    position: Optional["Position"] = None
    temperature: float = 20.0  # Degrees Celsius
    pressure: float = 101.3  # kPa
    humidity: float = 50.0  # Percent
    integrity_seal: bool = True
    vibration_level: float = 0.0  # Illustrative sensor units
    last_inspection: datetime = field(default_factory=datetime.now)


@dataclass
class Position:
    """A vessel position and its illustrative structural indicators."""
    id: str
    hold: int
    x: float
    y: float
    z: float
    structural_stress: float = 0.0  # MPa
    corrosion_level: float = 0.0  # Percent
    fire_resistance: float = 100.0  # Percent
    liquid_resistance: float = 100.0  # Percent


@dataclass
class SegregationRequirement:
    """A simplified pairwise segregation rule."""
    class1: str
    class2: str
    requirement: str
    min_distance: float
    different_hold: bool = False
    weather_sensitivity: float = 1.0
    vibration_tolerance: float = 1.0
    temperature_factor: float = 1.0


@dataclass
class VesselState:
    """The current simulated vessel and environmental state."""
    latitude: float = 40.7128
    longitude: float = -74.0060
    heading: float = 45.0  # Degrees
    speed: float = 15.0  # Knots
    roll: float = 0.0  # Degrees
    pitch: float = 0.0  # Degrees
    yaw: float = 0.0  # Degrees
    heave: float = 0.0  # Metres
    sea_state: int = 3
    wind_speed: float = 10.0  # Knots
    wave_height: float = 1.5  # Metres
    visibility: float = 10.0  # Nautical miles
    air_temperature: float = 15.0  # Degrees Celsius
    water_temperature: float = 12.0  # Degrees Celsius


@dataclass
class RiskAssessment:
    """A heuristic risk score and its accompanying indicators."""
    failure_probability: float
    risk_level: str
    time_to_failure: Optional[float]  # Illustrative hours, not a validated forecast
    contributing_factors: List[str]
    recommended_actions: List[str]
    confidence: float  # Heuristic data-quality indicator, not a statistical interval


## 1. Build the synthetic scenario

The scenario supplies three linked inputs: containers with dangerous-goods class labels, candidate vessel positions, and pairwise segregation requirements. Each position has coordinates and structural indicators. Each requirement either specifies a minimum Euclidean distance or requires the two containers to occupy different holds.

The labels and rules below are deliberately limited to this example. A real stowage and compliance system would require authoritative cargo data, vessel-specific geometry, and the applicable regulatory rules.

In [ ]:
def create_digital_twin_scenario():
    """Create the containers, positions, and rules for the teaching scenario."""
    container_configs = [
        ("EXP001", "1.4S"),
        ("EXP002", "1.1"),
        ("FLM001", "3"),
        ("FLM002", "3"),
        ("COR001", "8.1"),
        ("COR002", "8"),
        ("GAS001", "2.1"),
        ("GAS002", "2.3"),
        ("ORG001", "5.1"),
        ("TOX001", "6.1"),
        ("MISC01", "9"),
        ("RAD001", "4.1"),
    ]
    containers = [Container(container_id, imdg_class) for container_id, imdg_class in container_configs]

    positions = []
    for hold in range(1, 5):
        for row in range(3):
            for bay in range(4):
                positions.append(
                    Position(
                        id=f"H{hold}_R{row + 1}_B{bay + 1}",
                        hold=hold,
                        x=bay * 3.0,
                        y=row * 2.5,
                        z=(hold - 1) * 5.0,
                        structural_stress=random.uniform(0.1, 0.3),
                        corrosion_level=random.uniform(0.5, 2.0),
                        fire_resistance=random.uniform(85, 100),
                        liquid_resistance=random.uniform(90, 100),
                    )
                )

    segregation_requirements = [
        SegregationRequirement("1.1", "1.4S", "Away From", 5.0, False, 1.5, 0.8, 1.2),
        SegregationRequirement("1.1", "3", "Away From", 8.0, False, 1.8, 0.6, 1.3),
        SegregationRequirement("1.1", "8", "Separated From", 0.0, True, 2.0, 0.5, 1.5),
        SegregationRequirement("1.1", "2.1", "Away From", 8.0, False, 1.6, 0.7, 1.2),
        SegregationRequirement("1.1", "4.1", "Separated From", 0.0, True, 2.2, 0.4, 1.8),
        SegregationRequirement("1.4S", "3", "Away From", 4.0, False, 1.3, 0.9, 1.1),
        SegregationRequirement("1.4S", "8", "Separated From", 0.0, True, 1.7, 0.8, 1.3),
        SegregationRequirement("1.4S", "2.1", "Away From", 4.0, False, 1.4, 0.9, 1.1),
        SegregationRequirement("3", "8", "Away From", 3.0, False, 1.2, 1.0, 1.0),
        SegregationRequirement("3", "2.1", "No restriction", 0.0, False, 1.0, 1.0, 1.0),
        SegregationRequirement("3", "5.1", "Away From", 3.0, False, 1.1, 1.1, 1.0),
        SegregationRequirement("8", "2.3", "Away From", 3.0, False, 1.1, 1.0, 1.2),
        SegregationRequirement("8", "6.1", "Away From", 3.0, False, 1.0, 1.0, 1.1),
        SegregationRequirement("4.1", "3", "Away From", 10.0, False, 2.5, 0.3, 2.0),
        SegregationRequirement("4.1", "8", "Separated From", 0.0, True, 3.0, 0.2, 2.5),
    ]
    return containers, positions, segregation_requirements


containers, positions, seg_reqs = create_digital_twin_scenario()

print("Digital-twin scenario")
print(f"  Containers: {len(containers)}")
print(f"  Positions: {len(positions)} (4 holds × 3 rows × 4 bays)")
print(f"  Segregation requirements: {len(seg_reqs)}")

print("\nContainer distribution:")
class_counts = {}
for container in containers:
    class_counts[container.imdg_class] = class_counts.get(container.imdg_class, 0) + 1
for imdg_class, count in sorted(class_counts.items()):
    print(f"  Class {imdg_class}: {count} container(s)")

print("\nPosition distribution:")
hold_counts = {}
for position in positions:
    hold_counts[position.hold] = hold_counts.get(position.hold, 0) + 1
for hold, count in sorted(hold_counts.items()):
    print(f"  Hold {hold}: {count} positions")


## 2. Simulate vessel, environment, and cargo sensors

A synchronization cycle updates environmental conditions, derives vessel motion from the current weather inputs, and updates each placed container's sensor readings. The simple force calculation illustrates how vessel motion and container position could be combined, but it is not connected to the sensor-update rule in this example. Likewise, the forecast method demonstrates a possible component interface; the current simulation does not consume its output.

These component models use stochastic variation and simplified relationships. They demonstrate data flow, not physical prediction.

In [ ]:
class VesselDynamicsSimulator:
    """Simulate simplified vessel motion and cargo-force indicators."""

    def __init__(self):
        self.vessel_state = VesselState()
        self.time_step = 0

    def update_dynamics(self, weather_conditions: Dict[str, float]) -> VesselState:
        """Update motion and position using the supplied weather conditions."""
        state = self.vessel_state
        state.sea_state = int(weather_conditions.get("sea_state", 3))
        state.wind_speed = weather_conditions.get("wind_speed", 10.0)
        state.wave_height = weather_conditions.get("wave_height", 1.5)
        state.air_temperature = weather_conditions.get("air_temp", 15.0)

        wave_factor = state.wave_height / 3.0
        wind_factor = state.wind_speed / 25.0
        state.roll = float(np.clip(np.random.normal(0, 2.0 * wave_factor + wind_factor), -15, 15))
        state.pitch = float(np.clip(np.random.normal(0, 1.5 * wave_factor + 0.5 * wind_factor), -8, 8))
        state.heave = float(np.clip(np.random.normal(0, 0.5 * wave_factor), -2, 2))

        state.latitude += 0.01 * np.cos(np.radians(state.heading))
        state.longitude += 0.01 * np.sin(np.radians(state.heading))
        state.heading += np.random.normal(0, 0.5 * wind_factor)
        self.time_step += 1
        return state

    def calculate_cargo_forces(self, container: Container, position: Position) -> Dict[str, float]:
        """Return illustrative motion-related force indicators."""
        state = self.vessel_state
        roll_force = abs(state.roll) * container.weight * 0.1
        pitch_force = abs(state.pitch) * container.weight * 0.05
        heave_force = abs(state.heave) * container.weight * 0.2
        height_factor = 1.0 + position.z / 20.0
        total_force = (roll_force + pitch_force + heave_force) * height_factor
        return {
            "roll_force": roll_force,
            "pitch_force": pitch_force,
            "heave_force": heave_force,
            "total_force": total_force,
            "vibration_level": total_force * 0.1,
        }


class EnvironmentalMonitor:
    """Maintain current conditions and provide a simple forecast example."""

    def __init__(self):
        self.current_conditions = {
            "sea_state": 3,
            "wind_speed": 10.0,
            "wave_height": 1.5,
            "air_temp": 15.0,
            "visibility": 10.0,
            "barometric_pressure": 1013.0,
        }
        self.forecast_horizon = 72

    def update_conditions(self, vessel_state: VesselState) -> None:
        """Update the illustrative environment using the vessel's latitude."""
        conditions = self.current_conditions
        if vessel_state.latitude > 45:
            conditions["sea_state"] = min(
                8, conditions["sea_state"] + int(np.random.choice([-1, 0, 1]))
            )
            conditions["wind_speed"] = max(
                5, conditions["wind_speed"] + np.random.normal(0, 2)
            )
            conditions["wave_height"] = max(
                0.5, conditions["wave_height"] + np.random.normal(0, 0.3)
            )

        conditions["air_temp"] += np.random.normal(0, 0.5)
        conditions["sea_state"] = int(np.clip(conditions["sea_state"], 0, 12))
        conditions["wind_speed"] = float(np.clip(conditions["wind_speed"], 0, 60))
        conditions["wave_height"] = float(np.clip(conditions["wave_height"], 0, 15))
        conditions["air_temp"] = float(np.clip(conditions["air_temp"], -10, 35))

    def generate_forecast(self) -> List[Dict[str, float]]:
        """Generate a stochastic 72-step forecast example."""
        forecast = []
        current = self.current_conditions.copy()
        for hour in range(self.forecast_horizon):
            if hour < 24:
                trend = 0.1
            elif hour < 48:
                trend = 0.3
            else:
                trend = 0.5

            current["sea_state"] = float(
                np.clip(current["sea_state"] + np.random.normal(0, trend), 0, 12)
            )
            current["wind_speed"] = float(
                np.clip(current["wind_speed"] + np.random.normal(0, trend * 2), 0, 60)
            )
            current["wave_height"] = float(
                np.clip(current["wave_height"] + np.random.normal(0, trend * 0.5), 0, 15)
            )
            forecast.append(current.copy())
        return forecast


class CargoIntegrityTracker:
    """Update illustrative container sensor readings."""

    def __init__(self):
        self.sensor_network = True
        self.update_frequency = 30

    def update_container_sensors(
        self,
        container: Container,
        position: Position,
        vessel_state: VesselState,
        env_conditions: Dict[str, float],
    ) -> Container:
        """Update one container's temperature, pressure, humidity, and vibration."""
        ambient_temp = env_conditions["air_temp"]
        if container.imdg_class in ["8", "8.1"]:
            container.temperature = ambient_temp + np.random.uniform(5, 15)
        elif container.imdg_class == "3":
            container.temperature = ambient_temp + np.random.uniform(0, 5)
        else:
            container.temperature = ambient_temp + np.random.uniform(-2, 2)

        altitude_effect = position.z * 0.01
        weather_effect = (
            1013.0 - env_conditions.get("barometric_pressure", 1013.0)
        ) * 0.1
        container.pressure = (
            101.3 + altitude_effect + weather_effect + np.random.normal(0, 0.5)
        )
        container.humidity = float(
            np.clip(
                50 + np.random.normal(0, 10) + (ambient_temp - 15) * 2,
                20,
                90,
            )
        )
        vibration_base = abs(vessel_state.roll) * 0.5 + abs(vessel_state.pitch) * 0.3
        container.vibration_level = vibration_base + np.random.normal(0, 0.1)

        if container.vibration_level > 2.0 or container.temperature > 35:
            container.integrity_seal = (
                container.integrity_seal and random.random() > 0.1
            )
        return container


## 3. Assess risk from four components

The risk engine combines four heuristic components: structural stress and motion, pairwise segregation proximity, container and weather indicators, and implausible sensor readings. A weighted sum gives the overall score:

$$
R = 0.2R_{\mathrm{structural}} + 0.4R_{\mathrm{segregation}} + 0.3R_{\mathrm{environmental}} + 0.1R_{\mathrm{sensor}}.
$$

Each component score is intended to lie between 0 and 1. The weights sum to 1, so the weighted score also lies between 0 and 1. In this implementation, the score is named `failure_probability` for continuity with the scenario's decision-support interface, but it is not a calibrated probability. The thresholds label it Low, Medium, High, or Critical; the time-to-failure ranges are illustrative random values associated with those labels.

For each pair of placed containers, the engine searches for the first requirement whose two class labels match the pair in either order. If `different_hold` is true, being in the same hold adds risk. Otherwise, distance is calculated as

$$
d_{ij} = \sqrt{(x_i-x_j)^2+(y_i-y_j)^2+(z_i-z_j)^2}.
$$

When a distance requirement is violated, its base risk increases as the distance falls below the required minimum. Motion and sensor conditions then modify this score. Pairs without a matching requirement do not contribute to this component. The component is averaged over matched pairs, including matched pairs that satisfy their rule and therefore contribute zero.

In [ ]:
class PredictiveRiskEngine:
    """Combine simplified sensor and segregation indicators into a risk score."""

    def __init__(self):
        self.risk_models = {
            "structural_failure": self._assess_structural_failure_risk,
            "segregation_breach": self._assess_segregation_breach_risk,
            "environmental_compromise": self._assess_environmental_risk,
            "sensor_anomaly": self._assess_sensor_anomaly_risk,
        }

    def _assess_structural_failure_risk(
        self,
        containers: List[Container],
        positions: List[Position],
        vessel_state: VesselState,
    ) -> float:
        """Score structural stress and current vessel motion."""
        if not positions:
            return 0.0
        max_stress = max(position.structural_stress for position in positions)
        motion_factor = (abs(vessel_state.roll) + abs(vessel_state.pitch)) / 20.0
        return float(min(1.0, max_stress / 2.0 + motion_factor))

    def _assess_segregation_breach_risk(
        self,
        containers: List[Container],
        positions: List[Position],
        requirements: List[SegregationRequirement],
        vessel_state: VesselState,
    ) -> float:
        """Score matched container pairs against their simplified rules."""
        total_risk = 0.0
        risk_count = 0

        for container1, container2 in combinations(containers, 2):
            if container1.position is None or container2.position is None:
                continue

            applicable_req = next(
                (
                    req
                    for req in requirements
                    if (
                        req.class1 == container1.imdg_class
                        and req.class2 == container2.imdg_class
                    )
                    or (
                        req.class1 == container2.imdg_class
                        and req.class2 == container1.imdg_class
                    )
                ),
                None,
            )
            if applicable_req is None:
                continue

            pos1 = container1.position
            pos2 = container2.position
            distance = np.sqrt(
                (pos1.x - pos2.x) ** 2
                + (pos1.y - pos2.y) ** 2
                + (pos1.z - pos2.z) ** 2
            )

            if applicable_req.different_hold:
                risk = (
                    0.8 * applicable_req.weather_sensitivity
                    if pos1.hold == pos2.hold
                    else 0.0
                )
            elif distance < applicable_req.min_distance:
                distance_ratio = distance / applicable_req.min_distance
                risk = (1.0 - distance_ratio) * applicable_req.weather_sensitivity
            else:
                risk = 0.0

            motion_amplification = 1.0 + (
                abs(vessel_state.roll) + abs(vessel_state.pitch)
            ) / 30.0
            risk *= motion_amplification

            if container1.temperature > 30 or container2.temperature > 30:
                risk *= applicable_req.temperature_factor
            if container1.vibration_level > 2.0 or container2.vibration_level > 2.0:
                risk *= 2.0 - applicable_req.vibration_tolerance

            total_risk += min(1.0, risk)
            risk_count += 1

        return total_risk / risk_count if risk_count else 0.0

    def _assess_environmental_risk(
        self,
        containers: List[Container],
        env_conditions: Dict[str, float],
    ) -> float:
        """Average the temperature, vibration, seal, and weather indicators."""
        if not containers:
            return 0.0
        high_temp = sum(container.temperature > 35 for container in containers)
        temp_risk = min(1.0, high_temp / len(containers) * 2)
        high_vibration = sum(
            container.vibration_level > 2.5 for container in containers
        )
        vibration_risk = min(1.0, high_vibration / len(containers) * 2)
        compromised_seals = sum(
            not container.integrity_seal for container in containers
        )
        seal_risk = min(1.0, compromised_seals / len(containers) * 3)
        weather_risk = min(1.0, env_conditions.get("sea_state", 3) / 8.0)
        return float(np.mean([temp_risk, vibration_risk, seal_risk, weather_risk]))

    def _assess_sensor_anomaly_risk(self, containers: List[Container]) -> float:
        """Score readings outside the example's plausibility bounds."""
        if not containers:
            return 0.0
        anomalies = 0
        for container in containers:
            anomalies += int(container.temperature < -20 or container.temperature > 60)
            anomalies += int(container.pressure < 80 or container.pressure > 120)
            anomalies += int(container.humidity < 0 or container.humidity > 100)
        return min(1.0, anomalies / (len(containers) * 3))

    def comprehensive_risk_assessment(
        self,
        containers: List[Container],
        positions: List[Position],
        requirements: List[SegregationRequirement],
        vessel_state: VesselState,
        env_conditions: Dict[str, float],
    ) -> RiskAssessment:
        """Compute component scores, a weighted score, and explanatory indicators."""
        risks = {
            "structural_failure": self._assess_structural_failure_risk(
                containers, positions, vessel_state
            ),
            "segregation_breach": self._assess_segregation_breach_risk(
                containers, positions, requirements, vessel_state
            ),
            "environmental_compromise": self._assess_environmental_risk(
                containers, env_conditions
            ),
            "sensor_anomaly": self._assess_sensor_anomaly_risk(containers),
        }
        weights = {
            "structural_failure": 0.2,
            "segregation_breach": 0.4,
            "environmental_compromise": 0.3,
            "sensor_anomaly": 0.1,
        }
        score = sum(risks[name] * weights[name] for name in weights)

        if score < 0.1:
            risk_level = "Low"
            time_to_failure = None
        elif score < 0.3:
            risk_level = "Medium"
            time_to_failure = float(np.random.uniform(24, 72))
        elif score < 0.6:
            risk_level = "High"
            time_to_failure = float(np.random.uniform(6, 24))
        else:
            risk_level = "Critical"
            time_to_failure = float(np.random.uniform(1, 6))

        contributing_factors = [
            f"{name.replace('_', ' ').title()}: {value:.1%}"
            for name, value in risks.items()
            if value > 0.3
        ]
        recommended_actions = []
        if risks["segregation_breach"] > 0.3:
            recommended_actions.append(
                "Review container placement and consider re-stowage"
            )
        if risks["structural_failure"] > 0.3:
            recommended_actions.append(
                "Reduce vessel speed and change course to minimize motion"
            )
        if risks["environmental_compromise"] > 0.3:
            recommended_actions.append("Inspect container seals and cooling systems")
        if risks["sensor_anomaly"] > 0.2:
            recommended_actions.append("Calibrate or replace malfunctioning sensors")

        confidence = 1.0 - risks["sensor_anomaly"] * 0.5
        return RiskAssessment(
            failure_probability=score,
            risk_level=risk_level,
            time_to_failure=time_to_failure,
            contributing_factors=contributing_factors,
            recommended_actions=recommended_actions,
            confidence=confidence,
        )


## 4. Connect the components and compare response scenarios

The digital-twin system coordinates the component updates and stores the risk assessment for each synchronization cycle. It begins with a randomized placement heuristic: explosive-labelled containers are preferentially assigned to Hold 1 and corrosive-labelled containers to Holds 3 or 4 when those positions remain available. This placement is only an initial demonstration configuration; the risk engine assesses it but does not optimize or repair it.

For High or Critical scores, four what-if actions are considered. Each action has an assumed cost and an absolute score reduction. The residual score is

$$
R_s = \max(0, R - \Delta_s),
$$

where $R$ is the current heuristic score and $\Delta_s$ is the scenario's assumed reduction. The comparison uses

$$
C_s^{\mathrm{expected}} = C_s + R_s C_{\mathrm{incident}},
$$

where $C_s$ is the scenario cost and $C_{\mathrm{incident}}$ is the illustrative incident-cost assumption. This is a simple ranking rule, not a full decision analysis: it does not model uncertainty distributions, safety constraints, or the operational feasibility of an action.

In [ ]:
class DigitalTwinSystem:
    """Coordinate simulation updates, risk assessments, and scenario analysis."""

    def __init__(
        self,
        containers: List[Container],
        positions: List[Position],
        requirements: List[SegregationRequirement],
    ):
        self.containers = containers
        self.positions = positions
        self.requirements = requirements
        self.vessel_simulator = VesselDynamicsSimulator()
        self.env_monitor = EnvironmentalMonitor()
        self.cargo_tracker = CargoIntegrityTracker()
        self.risk_engine = PredictiveRiskEngine()
        self.current_time = datetime.now()
        self.sync_interval = 30
        self.risk_history = []
        self._place_containers_initially()

    def _place_containers_initially(self) -> None:
        """Assign each container a distinct position using a simple heuristic."""
        available_positions = self.positions.copy()
        for container in self.containers:
            if not available_positions:
                break
            if container.imdg_class in ["1.1", "1.4S"]:
                preferred = [p for p in available_positions if p.hold == 1]
            elif container.imdg_class in ["8", "8.1"]:
                preferred = [p for p in available_positions if p.hold in [3, 4]]
            else:
                preferred = []
            position = random.choice(preferred or available_positions)
            container.position = position
            available_positions.remove(position)

    def synchronize(self) -> Dict[str, Any]:
        """Run one environmental, motion, sensor, and risk update."""
        self.env_monitor.update_conditions(self.vessel_simulator.vessel_state)
        weather = {
            key: self.env_monitor.current_conditions[key]
            for key in ["sea_state", "wind_speed", "wave_height", "air_temp"]
        }
        vessel_state = self.vessel_simulator.update_dynamics(weather)

        for container in self.containers:
            if container.position is not None:
                self.cargo_tracker.update_container_sensors(
                    container,
                    container.position,
                    vessel_state,
                    self.env_monitor.current_conditions,
                )

        assessment = self.risk_engine.comprehensive_risk_assessment(
            self.containers,
            self.positions,
            self.requirements,
            vessel_state,
            self.env_monitor.current_conditions,
        )
        timestamp = self.current_time
        environmental = self.env_monitor.current_conditions.copy()
        self.risk_history.append(
            {
                "timestamp": timestamp,
                "risk_assessment": assessment,
                "vessel_state": vessel_state,
                "environmental": environmental,
            }
        )
        if len(self.risk_history) > 1000:
            self.risk_history = self.risk_history[-1000:]

        self.current_time += timedelta(seconds=self.sync_interval)
        return {
            "timestamp": timestamp,
            "risk_assessment": assessment,
            "vessel_state": vessel_state,
            "environmental": environmental,
            "sync_count": len(self.risk_history),
        }

    def generate_what_if_scenarios(
        self, risk_assessment: RiskAssessment
    ) -> List[Dict[str, Any]]:
        """Create illustrative response options for High or Critical scores."""
        if risk_assessment.risk_level not in ["High", "Critical"]:
            return []

        assumptions = [
            {
                "name": "Continue Current Route",
                "description": "Maintain current course and speed",
                "estimated_cost": 0,
                "risk_reduction": 0.0,
                "time_impact": 0,
            },
            {
                "name": "Speed Reduction and Course Change",
                "description": "Reduce speed by 50% and alter course to avoid severe weather",
                "estimated_cost": 25000,
                "risk_reduction": 0.4,
                "time_impact": 12,
            },
            {
                "name": "Emergency Re-stowage",
                "description": "Divert to a port for emergency container re-stowage",
                "estimated_cost": 150000,
                "risk_reduction": 0.9,
                "time_impact": 24,
            },
            {
                "name": "Enhanced Monitoring",
                "description": "Increase monitoring frequency and prepare crew intervention",
                "estimated_cost": 5000,
                "risk_reduction": 0.2,
                "time_impact": 2,
            },
        ]
        scenarios = []
        for assumption in assumptions:
            residual_score = max(
                0.0,
                risk_assessment.failure_probability - assumption["risk_reduction"],
            )
            scenarios.append(
                {
                    **assumption,
                    "residual_failure_score": residual_score,
                    "expected_total_cost": (
                        assumption["estimated_cost"]
                        + residual_score * INCIDENT_COST
                    ),
                }
            )
        return scenarios

    def run_simulation(self, duration_hours: int = 24) -> Dict[str, Any]:
        """Run the simulation and summarize the full run and retained history."""
        print(f"Starting digital-twin simulation: {duration_hours} hours")
        print("=" * 60)

        sync_cycles = (duration_hours * 3600) // self.sync_interval
        critical_events = []
        scenarios = []
        max_risk = 0.0

        for cycle in range(int(sync_cycles)):
            result = self.synchronize()
            assessment = result["risk_assessment"]
            max_risk = max(max_risk, assessment.failure_probability)

            if assessment.risk_level == "Critical":
                event = {
                    "timestamp": result["timestamp"],
                    "risk_assessment": assessment,
                    "cycle": cycle,
                }
                critical_events.append(event)

                if len(critical_events) == 1:
                    scenarios = self.generate_what_if_scenarios(assessment)
                    print(f"\nCRITICAL RISK DETECTED at {result['timestamp']}")
                    print(f"Heuristic risk score: {assessment.failure_probability:.1%}")
                    time_text = (
                        f"{assessment.time_to_failure:.1f} hours"
                        if assessment.time_to_failure is not None
                        else "Not estimated for this risk band"
                    )
                    print(f"Illustrative time-to-failure indicator: {time_text}")
                    print("\nContributing factors:")
                    for factor in assessment.contributing_factors:
                        print(f"  - {factor}")
                    print("\nRecommended actions:")
                    for action in assessment.recommended_actions:
                        print(f"  - {action}")
                    print("\nWhat-if scenario assumptions:")
                    for scenario in scenarios:
                        print(
                            f"  {scenario['name']}: "
                            f"${scenario['estimated_cost']:,.0f} action cost, "
                            f"{scenario['risk_reduction']:.0%} absolute score reduction, "
                            f"{scenario['time_impact']}h time impact"
                        )
                        print(
                            "    Illustrative expected total cost: "
                            f"${scenario['expected_total_cost']:,.0f}"
                        )

            if (cycle + 1) % 100 == 0:
                hours_elapsed = (cycle + 1) * self.sync_interval / 3600
                print(
                    f"Simulation progress: {hours_elapsed:.1f} hours, "
                    f"score: {assessment.failure_probability:.1%} "
                    f"({assessment.risk_level})"
                )

        return {
            "total_cycles": int(sync_cycles),
            "critical_events": critical_events,
            "max_risk": max_risk,
            "final_risk": (
                self.risk_history[-1]["risk_assessment"]
                if self.risk_history
                else None
            ),
            "scenarios": scenarios,
            "risk_history": self.risk_history.copy(),
        }


## 5. Run the 24-hour simulation

A 30-second synchronization interval gives the requested run length its number of update cycles. The system reports whole-run cycle count and maximum score, while the detailed record list contains no more than the most recent 1,000 cycles. Therefore, plots and statistics based on that list describe the retained window, not necessarily the entire run.

In [ ]:
print("IMDG segregation - digital-twin system simulation")
print("=" * 70)
print(f"Scenario: North Atlantic crossing with {len(containers)} dangerous-goods containers")
print("Digital-twin components: vessel dynamics, environment, cargo sensors, risk engine")
print("Synchronization interval: 30 seconds\n")

digital_twin = DigitalTwinSystem(containers, positions, seg_reqs)
simulation_result = digital_twin.run_simulation(duration_hours=24)

print("\n" + "=" * 70)
print("DIGITAL-TWIN SIMULATION SUMMARY")
print("=" * 70)
print(f"Total synchronization cycles: {simulation_result['total_cycles']}")
print(f"Critical events detected: {len(simulation_result['critical_events'])}")
print(f"Maximum heuristic risk score: {simulation_result['max_risk']:.1%}")
print(f"Detailed records retained: {len(simulation_result['risk_history'])}")

final_risk = simulation_result["final_risk"]
if final_risk is not None:
    print(
        f"Final heuristic risk score: {final_risk.failure_probability:.1%} "
        f"({final_risk.risk_level})"
    )
    print(f"Heuristic data-quality indicator: {final_risk.confidence:.1%}")

if simulation_result["scenarios"]:
    best_scenario = min(
        simulation_result["scenarios"],
        key=lambda scenario: scenario["expected_total_cost"],
    )
    print("\nLowest illustrative expected-cost scenario:")
    print(f"  {best_scenario['name']}")
    print(f"  Expected total cost: ${best_scenario['expected_total_cost']:,.0f}")
    print(f"  Assumed absolute score reduction: {best_scenario['risk_reduction']:.0%}")
    print(f"  Time impact: {best_scenario['time_impact']} hours")
else:
    print("\nNo response scenarios were generated because no Critical event occurred.")
    print("Scenario generation is triggered by a Critical event in this simulation.")


## 6. Visualize the retained simulation history

The first dashboard shows the retained risk-score trajectory, vessel motion, weather indicators, final container temperatures, risk-category counts, and any scenario comparison generated by the simulation. The second dashboard maps the final container locations in each hold. Point colour identifies the container class; marker size represents the illustrative vibration reading. Empty positions are shown in grey.

In [ ]:
def visualize_digital_twin_results(
    simulation_result: Dict[str, Any], digital_twin: DigitalTwinSystem
) -> None:
    """Plot the retained history, scenario comparison, and final hold layouts."""
    risk_history = simulation_result["risk_history"]
    if not risk_history:
        print("No retained records are available to visualize.")
        return

    timestamps = [record["timestamp"] for record in risk_history]
    elapsed_hours = [
        (timestamp - timestamps[0]).total_seconds() / 3600
        for timestamp in timestamps
    ]
    assessments = [record["risk_assessment"] for record in risk_history]
    scores = [assessment.failure_probability for assessment in assessments]
    vessel_states = [record["vessel_state"] for record in risk_history]
    environmental = [record["environmental"] for record in risk_history]

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))

    ax = axes[0, 0]
    ax.plot(elapsed_hours, scores, color="firebrick", linewidth=1.5)
    ax.fill_between(elapsed_hours, 0, scores, alpha=0.2, color="firebrick")
    for event in simulation_result["critical_events"]:
        if timestamps[0] <= event["timestamp"] <= timestamps[-1]:
            event_hour = (event["timestamp"] - timestamps[0]).total_seconds() / 3600
            ax.scatter(
                event_hour,
                event["risk_assessment"].failure_probability,
                color="darkred",
                marker="x",
                s=70,
                label="Critical event",
            )
    ax.set_xlabel("Elapsed time in retained window (hours)")
    ax.set_ylabel("Heuristic risk score")
    ax.set_title("Risk Score in Retained History")
    ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
    handles, labels = ax.get_legend_handles_labels()
    if handles:
        ax.legend(handles, labels)

    ax = axes[0, 1]
    ax.plot(elapsed_hours, [state.roll for state in vessel_states], label="Roll")
    ax.plot(elapsed_hours, [state.pitch for state in vessel_states], label="Pitch")
    ax.set_xlabel("Elapsed time in retained window (hours)")
    ax.set_ylabel("Motion (degrees)")
    ax.set_title("Simulated Vessel Motion")
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax = axes[0, 2]
    sea_line = ax.plot(
        elapsed_hours,
        [conditions["sea_state"] for conditions in environmental],
        color="teal",
        label="Sea state",
    )
    twin_axis = ax.twinx()
    wind_line = twin_axis.plot(
        elapsed_hours,
        [conditions["wind_speed"] for conditions in environmental],
        color="darkmagenta",
        label="Wind speed",
    )
    ax.set_xlabel("Elapsed time in retained window (hours)")
    ax.set_ylabel("Sea state (Beaufort)", color="teal")
    twin_axis.set_ylabel("Wind speed (knots)", color="darkmagenta")
    ax.set_title("Simulated Environmental Conditions")
    ax.grid(True, alpha=0.3)
    lines = sea_line + wind_line
    ax.legend(lines, [line.get_label() for line in lines], loc="upper left")

    ax = axes[1, 0]
    final_temperatures = [container.temperature for container in digital_twin.containers]
    ax.hist(final_temperatures, bins=12, color="orange", edgecolor="black", alpha=0.75)
    if final_temperatures:
        mean_temperature = float(np.mean(final_temperatures))
        ax.axvline(
            mean_temperature,
            color="red",
            linestyle="--",
            label=f"Mean: {mean_temperature:.1f} °C",
        )
        ax.legend()
    ax.set_xlabel("Temperature (°C)")
    ax.set_ylabel("Number of containers")
    ax.set_title("Final Container Temperatures")
    ax.grid(True, alpha=0.3, axis="y")

    ax = axes[1, 1]
    level_order = ["Low", "Medium", "High", "Critical"]
    level_counts = pd.Series(
        [assessment.risk_level for assessment in assessments]
    ).value_counts().reindex(level_order, fill_value=0)
    level_colors = {
        "Low": "green",
        "Medium": "gold",
        "High": "darkorange",
        "Critical": "red",
    }
    bars = ax.bar(
        level_counts.index,
        level_counts.values,
        color=[level_colors[level] for level in level_counts.index],
        alpha=0.75,
    )
    ax.set_xlabel("Risk category")
    ax.set_ylabel("Retained record count")
    ax.set_title("Risk Categories in Retained History")
    ax.grid(True, alpha=0.3, axis="y")
    total_records = int(level_counts.sum())
    for bar, count in zip(bars, level_counts.values):
        if total_records:
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                bar.get_height(),
                f"{count / total_records:.1%}",
                ha="center",
                va="bottom",
            )

    ax = axes[1, 2]
    scenarios = simulation_result["scenarios"]
    if scenarios:
        costs = [scenario["expected_total_cost"] for scenario in scenarios]
        reductions = [scenario["risk_reduction"] for scenario in scenarios]
        ax.scatter(costs, reductions, s=90, color="steelblue", alpha=0.8)
        for scenario, cost, reduction in zip(scenarios, costs, reductions):
            ax.annotate(
                scenario["name"],
                (cost, reduction),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=8,
            )
        best_index = int(np.argmin(costs))
        ax.scatter(
            costs[best_index],
            reductions[best_index],
            s=180,
            facecolors="none",
            edgecolors="red",
            linewidths=2,
            label="Lowest expected cost",
        )
        ax.legend()
        ax.set_xlabel("Illustrative expected total cost ($)")
        ax.set_ylabel("Assumed absolute score reduction")
        ax.set_title("What-if Scenario Cost Comparison")
        ax.grid(True, alpha=0.3)
    else:
        ax.text(
            0.5,
            0.5,
            "No scenarios generated",
            ha="center",
            va="center",
            transform=ax.transAxes,
        )
        ax.set_title("What-if Scenario Comparison")
        ax.set_axis_off()

    fig.suptitle("IMDG Segregation: Digital-Twin Simulation", fontsize=16)
    fig.tight_layout()
    plt.show()

    class_colors = {
        "1.1": "darkred",
        "1.4S": "red",
        "3": "orange",
        "8": "purple",
        "8.1": "darkviolet",
        "2.1": "green",
        "2.3": "darkgreen",
        "5.1": "brown",
        "6.1": "olive",
        "4.1": "pink",
        "9": "blue",
    }
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    container_by_position = {
        container.position.id: container
        for container in digital_twin.containers
        if container.position is not None
    }

    for hold_number, ax in enumerate(axes.flat, start=1):
        hold_positions = [
            position for position in digital_twin.positions
            if position.hold == hold_number
        ]
        for position in hold_positions:
            container = container_by_position.get(position.id)
            if container is None:
                ax.scatter(
                    position.x,
                    position.y,
                    s=100,
                    color="lightgray",
                    marker="s",
                    alpha=0.5,
                    edgecolors="black",
                )
                ax.text(position.x, position.y, "Empty", ha="center", va="center", fontsize=6)
                continue

            color = class_colors.get(container.imdg_class, "gray")
            marker_size = 100 + container.vibration_level * 50
            ax.scatter(
                position.x,
                position.y,
                s=marker_size,
                color=color,
                marker="s",
                alpha=0.8,
                edgecolors="black",
            )
            ax.text(
                position.x,
                position.y,
                f"{container.id}\nClass {container.imdg_class}",
                ha="center",
                va="center",
                fontsize=7,
            )

        ax.set_title(f"Hold {hold_number}: Final Configuration")
        ax.set_xlabel("X position (m)")
        ax.set_ylabel("Y position (m)")
        ax.grid(True, alpha=0.3)
        ax.set_aspect("equal")

    legend_handles = [
        Patch(facecolor=color, edgecolor="black", label=f"Class {imdg_class}")
        for imdg_class, color in class_colors.items()
    ]
    fig.legend(handles=legend_handles, loc="lower center", ncol=6, title="Container class")
    fig.suptitle("Final Container Locations by Hold", fontsize=14)
    fig.tight_layout(rect=(0, 0.08, 1, 0.95))
    plt.show()


visualize_digital_twin_results(simulation_result, digital_twin)


## 7. Review the simulation performance

This report separates whole-run information from statistics calculated over the retained records. If the simulation exceeds the 1,000-record retention limit, risk averages, category counts, and the final assessment describe only the retained window. The maximum score and event count are accumulated across the full run.

In [ ]:
def analyze_digital_twin_performance(
    simulation_result: Dict[str, Any], digital_twin: DigitalTwinSystem
) -> None:
    """Report whole-run counters and retained-window risk statistics."""
    history = simulation_result["risk_history"]
    print("\nDIGITAL-TWIN PERFORMANCE SUMMARY")
    print("=" * 50)
    print(f"Synchronization cycles in run: {simulation_result['total_cycles']}")
    print(f"Detailed records retained: {len(history)}")
    print(f"Synchronization interval: {digital_twin.sync_interval} seconds")
    if history:
        retained_hours = (
            len(history) * digital_twin.sync_interval / 3600
        )
        print(f"Approximate retained time span: {retained_hours:.2f} hours")

        scores = [record["risk_assessment"].failure_probability for record in history]
        print("\nRisk-score statistics for retained records:")
        print(f"  Mean: {np.mean(scores):.1%}")
        print(f"  Maximum in retained window: {np.max(scores):.1%}")
        print(f"  Minimum: {np.min(scores):.1%}")
        print(f"  Range: {np.max(scores) - np.min(scores):.1%}")

        final_assessment = simulation_result["final_risk"]
        if final_assessment is not None:
            print("\nFinal heuristic assessment:")
            print(f"  Category: {final_assessment.risk_level}")
            print(f"  Data-quality indicator: {final_assessment.confidence:.1%}")
            print(f"  Contributing factors: {len(final_assessment.contributing_factors)}")
            print(f"  Suggested actions: {len(final_assessment.recommended_actions)}")

    print("\nComponent configuration:")
    print("  Vessel dynamics simulator: initialized")
    print("  Environmental monitor: initialized")
    print("  Cargo sensor tracker: initialized")
    print(f"  Sensor network flag: {digital_twin.cargo_tracker.sensor_network}")
    print(f"  Risk component models: {len(digital_twin.risk_engine.risk_models)}")
    print(f"  Critical events in full run: {len(simulation_result['critical_events'])}")
    print(f"  Scenarios generated: {len(simulation_result['scenarios'])}")
    print("\nInterpretation:")
    print("  Scores and recommendations follow the illustrative rules defined in this notebook.")
    print("  They are not validated operational risk estimates or safety instructions.")


analyze_digital_twin_performance(simulation_result, digital_twin)


## 8. Role and limitations of this tier

### What the integrated simulation demonstrates

- A recurring synchronization cycle can connect vessel state, environmental inputs, cargo sensor readings, and a risk-assessment component.
- A weighted score can combine several indicators into a single, inspectable summary, while component-level values support explanation.
- What-if scenarios can make assumptions about action costs and score reductions explicit, then rank the resulting expected-cost estimates.
- A bounded history can limit memory use, provided that reports clearly distinguish retained-window statistics from whole-run counters.

### Limitations

- The initial placement is not optimized and has no feasibility or optimality guarantee.
- The risk score, thresholds, component weights, time-to-failure indicator, and confidence indicator are illustrative and are not calibrated against incident data.
- The simplified rules do not implement the full IMDG Code, vessel loading constraints, structural engineering, cargo compatibility, or operational procedures.
- The example does not integrate forecast output or cargo-force output into the synchronization and risk calculations.
- The scenario ranking assumes fixed action costs, fixed absolute score reductions, and a single incident-cost value. It does not establish that the lowest-cost option is safe or operationally feasible.
- Randomness without a fixed seed means that results vary between executions.

### Appropriate use

This tier is useful for teaching system integration, data flow, monitoring concepts, and transparent what-if calculations. Real operational deployment would require validated data and models, regulatory and engineering review, sensor-quality controls, uncertainty analysis, and human decision-making procedures.